In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

import lightgbm as lgb

from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
# !pip install lightgbm

In [5]:
df = pd.read_csv("data/전처리/1_1_법정동_일별_날씨_전력량.csv")
#df.info()

In [6]:
df.tail(5)

,전체코드,시군구코드,시군구명,법정동코드,법정동명,날짜,파워,평균기온(°C),최저기온(°C),최고기온(°C),...,평균 20cm 지중온도(°C),평균 30cm 지중온도(°C),0.5m 지중온도(°C),1.0m 지중온도(°C),1.5m 지중온도(°C),3.0m 지중온도(°C),5.0m 지중온도(°C),합계 대형증발량(mm),합계 소형증발량(mm),9-9강수(mm)
181403,1174011000,11740,강동구,11000,강일동,2024-01-24,115124.547,-8.0,-11.2,-3.7,...,0.0,0.8,2.3,4.7,7.6,13.6,17.1,1.2,1.7,0.0
181404,1174011000,11740,강동구,11000,강일동,2024-01-25,99907.500,-5.3,-9.7,-0.5,...,-0.3,0.6,2.2,4.6,7.6,13.5,17.0,1.4,2.0,0.0
181405,1174011000,11740,강동구,11000,강일동,2024-01-26,94818.512,-1.7,-6.9,4.0,...,-0.4,0.4,2.1,4.5,7.5,13.5,17.0,1.4,1.9,0.0
181406,1174011000,11740,강동구,11000,강일동,2024-01-27,86792.828,-0.5,-4.6,4.6,...,-0.4,0.4,2.0,4.5,7.5,13.4,16.9,1.4,2.1,0.0
181407,1174011000,11740,강동구,11000,강일동,2024-01-28,86389.477,-1.5,-5.0,3.8,...,-0.4,0.3,1.9,4.4,7.4,13.3,16.9,1.3,1.9,0.0


In [7]:
feature_cols = [
    # 지역
    "전체코드",

    # 날씨
    
    "최저기온(°C)", "0.5m 지중온도(°C)", "평균 증기압(hPa)","가조시간(hr)","평균 상대습도(%)",

    # 기간

    "lag_1d", "lag_7d"

]
target_col = "파워"

In [8]:
new_df = df[["날짜"] + feature_cols + [target_col]].dropna().reset_index(drop=True)

KeyError: "['lag_1d', 'lag_7d'] not in index"

In [ ]:
# =========================
# 1) 시간순 Train/Valid/Test 분할
# =========================
new_df = new_df.sort_values("날짜").reset_index(drop=True)

n = len(new_df)
test_size = int(n * 0.20)
valid_size = int(n * 0.10)

train_end = n - (valid_size + test_size)
valid_end = n - test_size

train_df = new_df.iloc[:train_end]
valid_df = new_df.iloc[train_end:valid_end]
test_df  = new_df.iloc[valid_end:]

In [ ]:
# =========================
# 2) X / y 분리
# =========================
drop_cols = [target_col, "날짜"]  # 날짜는 직접 입력 피처로 쓰지 않음(우린 파생변수로 이미 반영)
X_train = train_df.drop(columns=drop_cols)
y_train = train_df[target_col]

X_valid = valid_df.drop(columns=drop_cols)
y_valid = valid_df[target_col]

X_test = test_df.drop(columns=drop_cols)
y_test = test_df[target_col]

In [ ]:

# =========================
# 3) 범주형 처리 (코드형 컬럼은 category로)
# =========================
cat_cols = []
for c in ["전체코드", "시군구코드", "법정동코드"]:
    if c in X_train.columns:
        cat_cols.append(c)

for c in cat_cols:
    X_train[c] = X_train[c].astype("category")
    X_valid[c] = X_valid[c].astype("category")
    X_test[c]  = X_test[c].astype("category")

In [ ]:
# =========================
# 4) LightGBM Baseline 모델
# =========================
model = LGBMRegressor(
    objective="regression_l2",   # 🔥 반드시 변경

    n_estimators=20000,
    learning_rate=0.03,

    num_leaves=63,
    max_depth=10,
    min_child_samples=80,        # ↓ 완화
    min_split_gain=0.05,         # ↓ 완화

    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,

    reg_alpha=0.1,               # ↓
    reg_lambda=5.0,              # ↓

    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    eval_metric="rmse",
    categorical_feature=cat_cols if len(cat_cols) > 0 else "auto",
    callbacks=[
        # early stopping (너무 오래 돌지 않게)
        # verbose는 200번마다 한번씩 로그
        __import__("lightgbm").early_stopping(stopping_rounds=200, verbose=True),
        __import__("lightgbm").log_evaluation(period=200),
    ]
)

In [ ]:
# =========================
# 5) 평가 (Valid / Test)
# =========================
def eval_print(name, y_true, y_pred):
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    r2 = r2_score(y_true, y_pred)
    print(f"[{name}] RMSE: {rmse:,.3f}")
    print(f"[{name}] R2  : {r2:.6f}")
    return rmse, r2

pred_valid = model.predict(X_valid)
pred_test  = model.predict(X_test)

eval_print("VALID", y_valid, pred_valid)
eval_print("TEST",  y_test,  pred_test)

print("Best iteration:", model.best_iteration_)